#Afternoon Code Marathon: Weather and Trip Planner

##1: Pair workflow and plan

##2: Prepare Colab and Groq access

In [ ]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("Key loaded successfully:", api_key is not None)

Key loaded successfully: True


In [ ]:
%pip -q install requests groq langchain-core

In [ ]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "Enable GROQ_API_KEY in Colab Secrets."

GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_ID = "openai/gpt-oss-20b"  # Same hosted-Qwen model as the morning activity.

available_ids = {item.id for item in GROQ_CLIENT.models.list().data}
assert MODEL_ID in available_ids, (
    "Selected model unavailable. Check the available tool-capable models."
)
print("Groq client and selected model ready.")

##3: Explore the supplied attractions database

###3.1: Create marathon_db.py

In [ ]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        # TODO 1A: Restrict results to is_outdoor = 1.
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        # TODO 1B: Add a parameterized category filter.
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]


if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [ ]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


###3.2: TODO 1: Complete the SQL filters

In [ ]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]

if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [ ]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


In [ ]:
from marathon_db import initialize_db, find_attractions

initialize_db()

berlin_parks = find_attractions(
    "Berlin",
    category="park",
    outdoor_only=True
)

assert berlin_parks, "Expected outdoor parks in Berlin."
assert all(row["is_outdoor"] == 1 for row in berlin_parks)
assert all(row["category"] == "park" for row in berlin_parks)

stockholm_outdoor = find_attractions(
    "Stockholm",
    outdoor_only=True
)

assert stockholm_outdoor
assert all(row["is_outdoor"] == 1 for row in stockholm_outdoor)

assert find_attractions("Madrid") == []

print("Database filters passed.")

Database filters passed.


# Step-4

Create marathon_api.py

In [ ]:

%%writefile marathon_api.py
"""Open-Meteo daily forecast helper for future trip planning."""
from datetime import datetime, timezone
import requests

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


def get_forecast(latitude, longitude, days=7):
    """Return compact daily forecasts for a location."""

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        raise ValueError("Invalid latitude or longitude")

    if not 1 <= days <= 7:
        raise ValueError("days must be from 1 to 7")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "precipitation_probability_max,wind_speed_10m_max",
        "forecast_days": days,
        "timezone": "auto",
        "wind_speed_unit": "kmh"
    }

    response = requests.get(
        FORECAST_URL,
        params=params,
        timeout=20
    )
    response.raise_for_status()

    data = response.json()
    daily = data["daily"]

    observations = []

    for index, date_string in enumerate(daily["time"]):
        observations.append({
            "date": date_string,
            "rain_probability_max_pct":
                daily["precipitation_probability_max"][index],
            "wind_speed_max_kmh":
                daily["wind_speed_10m_max"][index]
        })

    return {
        "timezone": data.get("timezone"),
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": "Open-Meteo Forecast API",
        "days": observations
    }


if __name__ == "__main__":
    sample = get_forecast(52.515, 13.351, days=3)
    print(sample)


Overwriting marathon_api.py


In [ ]:

!python marathon_api.py


{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-10-29T12:35:04.080583+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-10-29', 'rain_probability_max_pct': 8, 'wind_speed_max_kmh': 7.6}, {'date': '2026-10-30', 'rain_probability_max_pct': 3, 'wind_speed_max_kmh': 8.2}, {'date': '2026-10-31', 'rain_probability_max_pct': 3, 'wind_speed_max_kmh': 9.5}]}


Test the forecast function

In [ ]:

from marathon_api import get_forecast

forecast = get_forecast(52.515, 13.351, days=3)

assert len(forecast["days"]) == 3
assert forecast["timezone"]

for day in forecast["days"]:
    assert isinstance(
        day["rain_probability_max_pct"], (int, float)
    )
    assert isinstance(
        day["wind_speed_max_kmh"], (int, float)
    )
    assert 0 <= day["rain_probability_max_pct"] <= 100
    assert day["wind_speed_max_kmh"] >= 0

print("Forecast extraction passed.")
print(forecast)


Forecast extraction passed.
{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-10-29T12:35:04.241884+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-10-29', 'rain_probability_max_pct': 8, 'wind_speed_max_kmh': 7.6}, {'date': '2026-10-30', 'rain_probability_max_pct': 3, 'wind_speed_max_kmh': 8.2}, {'date': '2026-10-31', 'rain_probability_max_pct': 3, 'wind_speed_max_kmh': 9.5}]}


# Step-5


Create marathon_tools.py

In [ ]:

%%writefile marathon_tools.py
"""LangChain wrappers around the approved database and API functions."""

from langchain_core.tools import tool
from marathon_db import find_attractions
from marathon_api import get_forecast


@tool
def find_attractions_tool(
    city: str,
    category: str | None = None,
    outdoor_only: bool = True,
    limit: int = 5,
) -> list[dict]:
    """Search seeded attractions in Stockholm, Gothenburg, Berlin, or Munich.

    Use for choosing real candidate places. Returns names, approximate
    coordinates, categories, indoor/outdoor flags, and descriptions.
    Does not provide opening hours or real-time availability.
    """
    return find_attractions(city, category, outdoor_only, limit)


@tool
def get_forecast_tool(
    latitude: float,
    longitude: float,
    days: int = 7
) -> dict:
    """Get a future daily forecast for coordinates from Open-Meteo.

    Returns local dates, daily maximum precipitation probability (%),
    and daily maximum 10-m wind speed (km/h). Call after obtaining
    coordinates from a returned attractions database row.
    """
    return get_forecast(latitude, longitude, days)


APPROVED_TOOLS = [
    find_attractions_tool,
    get_forecast_tool
]

TOOL_REGISTRY = {
    item.name: item for item in APPROVED_TOOLS
}


def groq_schema(item):
    """Convert a LangChain tool into a Groq-compatible function schema."""
    return {
        "type": "function",
        "function": {
            "name": item.name,
            "description": item.description,
            "parameters": item.args_schema.model_json_schema()
        }
    }


TOOL_SCHEMAS = [
    groq_schema(item) for item in APPROVED_TOOLS
]


Overwriting marathon_tools.py


Test both tools without an LLM

In [ ]:

from marathon_tools import (
    find_attractions_tool,
    get_forecast_tool,
    TOOL_SCHEMAS
)

# Test the database tool
attractions = find_attractions_tool.invoke({
    "city": "Stockholm",
    "outdoor_only": True,
    "limit": 3
})

assert attractions
assert all(place["is_outdoor"] == 1 for place in attractions)

# Use a real returned attraction to test the forecast tool
place = attractions[0]

weather = get_forecast_tool.invoke({
    "latitude": place["latitude"],
    "longitude": place["longitude"],
    "days": 3
})

assert len(weather["days"]) == 3
assert all(
    day["rain_probability_max_pct"] is not None
    for day in weather["days"]
)

print(
    "Registered tool names:",
    [item["function"]["name"] for item in TOOL_SCHEMAS]
)
print("Attraction:", place["name"])
print("Forecast sample:", weather["days"][0])
print("Both tool tests passed.")


Registered tool names: ['find_attractions_tool', 'get_forecast_tool']
Attraction: Djurgården
Forecast sample: {'date': '2026-10-29', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 13.3}
Both tool tests passed.


# Step-6


Install and check the dependencies

In [ ]:
%pip -q install groq langchain-core requests

In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")
assert api_key, "Add GROQ_API_KEY in Colab Secrets first."

client = Groq(api_key=api_key)

# Check whether the assignment's suggested model is available.
models = client.models.list()
available_models = [model.id for model in models.data]

MODEL = "qwen/qwen3.8-27b"

if MODEL in available_models:
    print("Model available:", MODEL)
else:
    print("Suggested model is not available. Available Qwen models:")
    print([name for name in available_models if "qwen" in name.lower()])

Model available: qwen/qwen3.8-27b


In [ ]:
%%writefile marathon_agent.py
import json
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
from groq import Groq

from marathon_tools import TOOL_SCHEMAS, TOOL_REGISTRY

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

SYSTEM_PROMPT = """
You plan outdoor trips using the attraction database and weather forecast.

Use real attraction records and forecast results. Never invent data.

A suitable day must have BOTH:
- Maximum precipitation probability below 30%.
- Maximum wind speed below 35 km/h.

If no forecast day meets both rules, state that clearly.
Explain the values used. Never claim missing weather data is suitable.
The database does not provide opening hours.
Keep the final answer short.
"""

def run_agent(user_request, max_iterations=5):
    api_key = os.getenv("GROQ_API_KEY") or (userdata.get("GROQ_API_KEY") if userdata else None)
    if not api_key:
        raise RuntimeError("Add GROQ_API_KEY in Colab Secrets.")

    client = Groq(api_key=api_key)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_request},
    ]
    trace = []

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.2,
            max_completion_tokens=300,
            parallel_tool_calls=False,
        )

        msg = response.choices[0].message
        calls = msg.tool_calls

        trace.append({
            "iteration": iteration + 1,
            "event": "model_response",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": c.id,
                    "name": c.function.name,
                    "arguments": c.function.arguments,
                }
                for c in (calls or [])
            ],
        })

        if not calls:
            return {
                "answer": msg.content or "",
                "trace": trace,
            }

        messages.append(msg.model_dump(exclude_none=True))

        for call in calls:
            name = call.function.name

            if name not in TOOL_REGISTRY:
                result = {"error": f"Tool not approved: {name}"}
            else:
                try:
                    args = json.loads(call.function.arguments)
                    result = TOOL_REGISTRY[name].invoke(args)
                except Exception as exc:
                    result = {"error": f"{type(exc).__name__}: {exc}"}

            trace.append({
                "iteration": iteration + 1,
                "event": "tool_result",
                "tool": name,
                "result": result,
            })

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(
                    result, ensure_ascii=False, default=str
                ),
            })

    return {
        "answer": "Tool-call limit reached. Inspect the trace and retry.",
        "trace": trace,
    }


In [ ]:

import importlib
import marathon_agent

importlib.reload(marathon_agent)

print("Current model:", marathon_agent.MODEL)
print(
    "Current completion limit:",
    marathon_agent.run_agent.__code__.co_consts
)


Current model: qwen/qwen3.8-27b
Current completion limit: (None, 'GROQ_API_KEY', 'Add GROQ_API_KEY in Colab Secrets.', ('api_key',), 'system', ('role', 'content'), 'user', 'auto', 0.2, 300, 'none', False, ('model', 'messages', 'tools', 'tool_choice', 'temperature', 'max_completion_tokens', 'reasoning_effort', 'parallel_tool_calls'), 0, 1, 'model_response', ('id', 'name', 'arguments'), ('iteration', 'event', 'content', 'tool_calls'), '', ('answer', 'trace'), True, ('exclude_none',), 'error', 'Tool not approved: ', ': ', 'tool_result', ('iteration', 'event', 'tool', 'result'), 'tool', ('ensure_ascii', 'default'), ('role', 'tool_call_id', 'name', 'content'), 'Tool-call limit reached. Inspect the trace and retry.')


In [ ]:
# The agent already defaults to openai/gpt-oss-20b.
import marathon_agent
print("Agent model:", marathon_agent.MODEL)


In [ ]:
# Model can be overridden with GROQ_MODEL if your Groq account supports another tool-capable model.
import marathon_agent
print("Current model:", marathon_agent.MODEL)


In [ ]:
result = marathon_agent.run_agent(
    "Find one outdoor attraction in Stockholm and check the forecast. "
    "Recommend a suitable date if both weather rules are met. "
    "Keep your answer brief."
)

print("FINAL RECOMMENDATION")
print(result["answer"])

print("\nTRACE SUMMARY")
for event in result["trace"]:
    print(
        "Iteration:", event["iteration"],
        "| Event:", event["event"],
        "| Tool:", event.get("tool", "")
    )

FINAL RECOMMENDATION
**Attraction:** Djurgården (Stockholm) – an outdoor park.  
**Best date:** **2026‑10‑29**  
- Precipitation probability: **0 %** (below 30 %)  
- Wind speed: **13.3 km/h** (below 35 km/h)  

All other forecast days also meet the rules, but this is the earliest suitable day.

TRACE SUMMARY
Iteration: 1 | Event: model_response | Tool: 
Iteration: 1 | Event: tool_result | Tool: find_attractions_tool
Iteration: 2 | Event: model_response | Tool: 
Iteration: 2 | Event: tool_result | Tool: get_forecast_tool
Iteration: 3 | Event: model_response | Tool: 


In [ ]:
print("FINAL ANSWER:")
print(result["answer"])

print("\nTRACE EVENTS:")
for event in result["trace"]:
    print(
        f"Iteration {event['iteration']} | "
        f"{event['event']} | "
        f"{event.get('tool', '')}"
    )

last_event = result["trace"][-1]
print("\nLast event:", last_event["event"])

FINAL ANSWER:
**Attraction:** Djurgården (Stockholm) – an outdoor park.  
**Best date:** **2026‑10‑29**  
- Precipitation probability: **0 %** (below 30 %)  
- Wind speed: **13.3 km/h** (below 35 km/h)  

All other forecast days also meet the rules, but this is the earliest suitable day.

TRACE EVENTS:
Iteration 1 | model_response | 
Iteration 1 | tool_result | find_attractions_tool
Iteration 2 | model_response | 
Iteration 2 | tool_result | get_forecast_tool
Iteration 3 | model_response | 

Last event: model_response


In [ ]:
import json

called_tools = {
    event["tool"]
    for event in result["trace"]
    if event["event"] == "tool_result"
}

assert "find_attractions_tool" in called_tools
assert "get_forecast_tool" in called_tools
assert result["trace"][-1]["event"] == "model_response"

with open("stockholm_trace.json", "w", encoding="utf-8") as f:
    json.dump(
        result["trace"],
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Step 6 checks passed.")
print("Tools used:", called_tools)
print("Trace saved to stockholm_trace.json")

Step 6 checks passed.
Tools used: {'find_attractions_tool', 'get_forecast_tool'}
Trace saved to stockholm_trace.json


##7: Investigate and evaluate the assistant

###7.1: Run test cases

In [ ]:
import json
from marathon_agent import run_agent

test_requests = {
    "Stockholm": (
        "Find an outdoor attraction in Stockholm and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Berlin": (
        "Find an outdoor attraction in Berlin and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Madrid": (
        "Find an outdoor attraction in Madrid. If the database has "
        "no matching attractions, report that instead of inventing one."
    ),
    "Munich": (
        "Find an outdoor park in Munich and check the forecast. "
        "Recommend a day only if both weather thresholds are satisfied."
    ),
}

all_traces = {}
all_results = {}

for city, request in test_requests.items():
    print(f"\n{'=' * 50}")
    print(f"Testing: {city}")
    print("=" * 50)

    try:
        result = run_agent(request)
        all_results[city] = result
        all_traces[city] = result["trace"]

        print("Final answer:")
        print(result["answer"])

        tool_calls = [
            event["tool"]
            for event in result["trace"]
            if event["event"] == "tool_result"
        ]

        print("Tools called:", tool_calls)
        print("Number of tool calls:", len(tool_calls))

    except Exception as exc:
        print("Test error:", str(exc))
        all_results[city] = {"error": str(exc)}
        all_traces[city] = []

with open("marathon_traces.json", "w", encoding="utf-8") as file:
    json.dump(
        all_traces,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("\nSaved all traces to marathon_traces.json")


Testing: Stockholm
Final answer:
**Outdoor attraction:** Djurgården (park, Stockholm)  
**Best forecast day:** 2026‑10‑29  
- Rain probability: 0 % (below 30 %)  
- Wind speed: 13.7 km/h (below 35 km/h)
Tools called: ['find_attractions_tool', 'get_forecast_tool']
Number of tool calls: 2

Testing: Berlin
Final answer:
**Outdoor attraction:** Tempelhofer Feld (park, 52.473 N, 13.405 E)

**Best day to visit:** **2026‑10‑29**  
- Rain probability max: **3 %** (below 30 %)  
- Wind speed max: **7.9 km/h** (below 35 km/h)  

All forecast days for the next week meet the criteria, but 2026‑10‑29 is the earliest suitable date.
Tools called: ['find_attractions_tool', 'get_forecast_tool']
Number of tool calls: 2

Testing: Madrid
Final answer:
No outdoor attractions are available in the database for Madrid.
Tools called: []
Number of tool calls: 0

Testing: Munich
Final answer:
**Recommended day:** 2026‑10‑29  
**Location:** English Garden, Munich  

**Weather values used:**  
- Precipitation pro

In [ ]:
import pandas as pd

# Compute evaluation from actual traces rather than hard-coded, contradictory counts.
rows = []
for city, result in all_results.items():
    trace = all_traces.get(city, [])
    calls = [e for e in trace if e.get("event") == "tool_result"]
    names = [e.get("tool", "") for e in calls]
    rows.append({
        "Request": city,
        "Tools called (in order)": " → ".join(names) or "None",
        "Tool calls": len(calls),
        "Repeated tool names": len(names) - len(set(names)),
        "Model produced answer?": bool(result.get("answer")),
        "Database checked?": "find_attractions_tool" in names,
        "Weather checked?": "get_forecast_tool" in names,
        "Needs manual grounding review?": True,
        "Error": result.get("error", ""),
    })
evaluation_table = pd.DataFrame(rows)
display(evaluation_table)
evaluation_table.to_csv("evaluation_table.csv", index=False)
print("Saved evaluation_table.csv")


### 7.3 Guided questions (observed results from the supplied notebook)

1. **First tool:** Stockholm, Berlin and Munich first called `find_attractions_tool`, appropriately obtaining candidate attractions and coordinates before weather lookup. Madrid did not call a tool, which was inappropriate for establishing a database absence.

2. **Both tools:** Stockholm, Berlin and Munich each used the attractions database and the weather forecast tool. Madrid did not need a forecast, but it did need a database lookup to substantiate the negative claim.

3. **Call counts:** In the four-case execution output, Stockholm had one attractions lookup and one weather lookup; Berlin likewise had one of each; Munich likewise had one of each; Madrid made no calls. These are distinct from the separately recorded single-Stockholm demonstration. Repeated calls with identical arguments are not shown in that four-case execution. Repetition can be justified after errors, with fresh data, or to compare distinct candidates, but not merely to restate the same result.

4. **Grounding:** Djurgården, Tempelhofer Feld and English Garden are present in the seeded SQLite dataset. The outputs reported both rain and wind values for the chosen date. Because full machine-readable test traces are not embedded in this exported notebook, precise agreement with each raw Open-Meteo payload cannot be independently audited from the outputs alone. The Madrid claim was not tool-verified in the agent's test run, although the direct database-tool demonstration returned no rows.


# Step-8


Demonstrate one successful recommendation

In [ ]:

from marathon_agent import run_agent
import json

stockholm_result = run_agent(
    "Find an outdoor attraction in Stockholm. "
    "Check the forecast and recommend a day only if rain probability "
    "is below 30% and wind speed is below 35 km/h."
)

print("FINAL ANSWER")
print(stockholm_result["answer"])

print("\nTOOL EXECUTION EVIDENCE")
for event in stockholm_result["trace"]:
    if event["event"] == "tool_result":
        print(f"\nTool: {event['tool']}")
        print(json.dumps(
            event["result"],
            indent=2,
            ensure_ascii=False,
            default=str
        ))


FINAL ANSWER
**Recommended day:** 2026‑10‑29  
**Attraction:** Djurgården, Stockholm  

- **Precipitation probability:** 0 % (below 30 %)  
- **Wind speed:** 13.7 km/h (below 35 km/h)  

Both criteria are met, so this day is suitable for an outdoor visit.

TOOL EXECUTION EVIDENCE

Tool: find_attractions_tool
[
  {
    "id": 1,
    "name": "Djurgården",
    "city": "Stockholm",
    "country": "Sweden",
    "category": "park",
    "is_outdoor": 1,
    "latitude": 59.325,
    "longitude": 18.103,
    "description": "Large area for outdoor walks and sightseeing."
  },
  {
    "id": 2,
    "name": "Skansen Open-Air Museum",
    "city": "Stockholm",
    "country": "Sweden",
    "category": "museum",
    "is_outdoor": 1,
    "latitude": 59.324,
    "longitude": 18.101,
    "description": "Open-air cultural attractions and historical buildings."
  }
]

Tool: get_forecast_tool
{
  "timezone": "Europe/Stockholm",
  "retrieved_at_utc": "2026-10-29T12:35:43.459037+00:00",
  "source": "Open-Meteo F

Demonstrate a failure or missing-data case

In [ ]:

from marathon_tools import find_attractions_tool

madrid_result = find_attractions_tool.invoke({
    "city": "Madrid",
    "outdoor_only": True,
    "limit": 5
})

print("Madrid database result:")
print(madrid_result)

if not madrid_result:
    print("Expected result: no matching attractions found.")


Madrid database result:
[]
Expected result: no matching attractions found.


1. **Identify two tools that could be useful in a possible course project.**

   Two useful tools are a database search tool to find attractions and a weather forecast tool to retrieve precipitation probability and wind speed.

2. **Identify a suitable database or external API for the project.**

   SQLite is a suitable database for storing attraction names, cities, categories, coordinates, and outdoor information. The Open-Meteo Forecast API is suitable for retrieving weather forecasts.

3. **Propose one user request that requires combining at least two tools.**

   An example user request is: “Find an outdoor attraction in Stockholm and recommend a day when the rain probability is below 30% and wind speed is below 35 km/h.” This request requires the agent to use the database tool to find an attraction and the weather tool to check the forecast.

4. **Describe one potential tool-use failure and an evaluation method for detecting it.**

   One potential failure is that the database may not contain attractions for the requested city, or the weather API may fail to return forecast data. To detect these failures, inspect the tool execution trace, check whether the tools returned valid results, and verify that the final recommendation is supported by the database and weather data. Both weather thresholds must be satisfied on the same date.

### Submission note
Run the evaluation cells in Colab with your own GROQ_API_KEY secret to generate **actual raw** `marathon_traces.json` and `evaluation_table.csv`. Do not submit the secret. The standalone packaged trace JSON is an explicitly marked summary reconstructed from displayed notebook outputs, not a raw tool trace.
